In [1]:
# ==========================================================
# TRAFFIC DEMAND PREDICTION — IMPROVED PIPELINE
# Key changes: geohash prefix features, cyclic day encoding,
# target encoding (OOF-safe), aggregation features,
# tuned hyperparams, + LightGBM blend
# ==========================================================

!pip install -q catboost lightgbm pygeohash optuna

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pygeohash as pgh

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

TRAIN_PATH = "/kaggle/input/datasets/rohanthakar/traffic-demand-prediction-gridlock/dataset/train.csv"
TEST_PATH  = "/kaggle/input/datasets/rohanthakar/traffic-demand-prediction-gridlock/dataset/test.csv"

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

# ==========================================================
# FEATURE ENGINEERING
# ==========================================================

def create_features(df):
    df = df.copy()

    # --- Timestamp ---
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="%H:%M", errors="coerce")
    df["hour"]   = df["timestamp"].dt.hour
    df["minute"] = df["timestamp"].dt.minute

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    # NEW: cyclic day encoding (if day is numeric 0-6 or 1-7)
    df["day_sin"] = np.sin(2 * np.pi * df["day"] / 7)
    df["day_cos"] = np.cos(2 * np.pi * df["day"] / 7)

    # NEW: combined time block (helps tree models cut day×hour interactions)
    df["time_block"] = df["hour"] * 7 + df["day"]

    # --- Geohash decode ---
    lats, lons = [], []
    for geo in df["geohash"]:
        try:
            lat, lon = pgh.decode(geo)
            lats.append(float(lat)); lons.append(float(lon))
        except:
            lats.append(np.nan); lons.append(np.nan)
    df["latitude"]  = lats
    df["longitude"] = lons

    # NEW: geohash precision levels — broader area features
    df["geohash4"] = df["geohash"].str[:4]
    df["geohash5"] = df["geohash"].str[:5]
    df["geohash6"] = df["geohash"].str[:6]

    # --- Temperature ---
    df["temp_sq"]   = df["Temperature"] ** 2
    df["lane_temp"] = df["NumberofLanes"] * df["Temperature"]

    # NEW: bins for interpretable interactions
    df["hour_bin"] = pd.cut(
        df["hour"],
        bins=[-1, 6, 10, 16, 20, 24],
        labels=["night", "morning", "midday", "evening", "latenight"]
    ).astype(str)

    return df

train = create_features(train)
test  = create_features(test)

# ==========================================================
# MISSING VALUES
# ==========================================================

temp_median = train["Temperature"].median()
train["Temperature"] = train["Temperature"].fillna(temp_median)
test["Temperature"]  = test["Temperature"].fillna(temp_median)

cat_cols = ["RoadType", "LargeVehicles", "Landmarks", "Weather", "geohash",
            "geohash4", "geohash5", "geohash6", "hour_bin"]
for col in cat_cols:
    train[col] = train[col].fillna("Unknown")
    test[col]  = test[col].fillna("Unknown")

# ==========================================================
# GEOHASH FREQUENCY ENCODING
# ==========================================================

for gh_col in ["geohash", "geohash4", "geohash5", "geohash6"]:
    freq = train[gh_col].value_counts()
    train[f"{gh_col}_freq"] = train[gh_col].map(freq).fillna(0)
    test[f"{gh_col}_freq"]  = test[gh_col].map(freq).fillna(0)

# ==========================================================
# LOCATION CLUSTERING (more clusters)
# ==========================================================

coords = pd.concat([train[["latitude","longitude"]], test[["latitude","longitude"]]]).fillna(
    pd.concat([train[["latitude","longitude"]], test[["latitude","longitude"]]]).median()
)
km = KMeans(n_clusters=30, random_state=42, n_init=10).fit(coords)
train["location_cluster"] = km.predict(train[["latitude","longitude"]].fillna(coords.median()))
test["location_cluster"]  = km.predict(test[["latitude","longitude"]].fillna(coords.median()))

# ==========================================================
# OOF TARGET ENCODING  ← biggest expected lift
# Encode: RoadType, geohash, geohash4, Weather by hour/day
# Uses only train fold data to avoid leakage
# ==========================================================

TARGET = "demand"
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

def oof_target_encode(train_df, test_df, group_cols, target_col, alpha=20):
    """
    Smoothed mean target encoding. alpha controls regularisation
    (higher = more shrinkage toward global mean).
    Returns encoded train (OOF-safe) and test columns.
    """
    global_mean = train_df[target_col].mean()
    col_name    = "_x_".join(group_cols) + "_te"

    train_enc = np.zeros(len(train_df))
    test_enc  = np.zeros(len(test_df))

    for fold, (tr_idx, val_idx) in enumerate(kf.split(train_df)):
        # Compute stats on training fold only
        stats = (
            train_df.iloc[tr_idx]
            .groupby(group_cols)[target_col]
            .agg(["mean", "count"])
            .reset_index()
        )
        stats["smoothed"] = (
            (stats["mean"] * stats["count"] + global_mean * alpha)
            / (stats["count"] + alpha)
        )
        # Apply to validation fold
        val_df = train_df.iloc[val_idx][group_cols].copy()
        merged = val_df.merge(stats[group_cols + ["smoothed"]], on=group_cols, how="left")
        train_enc[val_idx] = merged["smoothed"].fillna(global_mean).values

    # Full fit for test
    stats_full = (
        train_df.groupby(group_cols)[target_col]
        .agg(["mean", "count"])
        .reset_index()
    )
    stats_full["smoothed"] = (
        (stats_full["mean"] * stats_full["count"] + global_mean * alpha)
        / (stats_full["count"] + alpha)
    )
    test_merged = test_df[group_cols].merge(
        stats_full[group_cols + ["smoothed"]], on=group_cols, how="left"
    )
    test_enc = test_merged["smoothed"].fillna(global_mean).values

    return train_enc, test_enc, col_name


# Define which group combos to encode
encode_groups = [
    ["RoadType"],
    ["RoadType", "hour"],
    ["RoadType", "day"],
    ["geohash4"],
    ["geohash4", "hour"],
    ["geohash5"],
    ["Weather", "hour"],
    ["location_cluster", "hour"],
]

for groups in encode_groups:
    tr_enc, te_enc, col = oof_target_encode(train, test, groups, TARGET)
    train[col] = tr_enc
    test[col]  = te_enc

print("Target encoding done. New features:", [g for g in train.columns if g.endswith("_te")])

# ==========================================================
# AGGREGATION FEATURES  ← second biggest lift
# Mean/std of demand per key group, computed OOF-safe
# ==========================================================

agg_groups = [
    (["geohash", "hour"],         ["mean", "std"]),
    (["geohash4", "hour"],        ["mean", "std"]),
    (["RoadType", "hour"],        ["mean"]),
    (["RoadType", "day", "hour"], ["mean"]),
    (["location_cluster", "hour"],["mean", "std"]),
]

for groups, aggs in agg_groups:
    for agg_fn in aggs:
        col = "_".join(groups) + f"_{agg_fn}"
        oof_vals = np.zeros(len(train))

        for _, (tr_idx, val_idx) in enumerate(kf.split(train)):
            stats = (
                train.iloc[tr_idx]
                .groupby(groups)[TARGET]
                .agg(agg_fn)
                .reset_index()
                .rename(columns={TARGET: col})
            )
            merged = train.iloc[val_idx][groups].merge(stats, on=groups, how="left")
            oof_vals[val_idx] = merged[col].fillna(train[TARGET].mean()).values

        train[col] = oof_vals

        # Full fit → test
        stats_full = (
            train.groupby(groups)[TARGET]
            .agg(agg_fn)
            .reset_index()
            .rename(columns={TARGET: col})
        )
        test_merged = test[groups].merge(stats_full, on=groups, how="left")
        test[col] = test_merged[col].fillna(train[TARGET].mean()).values

print("Aggregation features done.")

# ==========================================================
# PREPARE DATA
# ==========================================================

test_ids = test["Index"]
drop_cols = [TARGET, "Index", "timestamp"]

X      = train.drop(columns=drop_cols)
y      = train[TARGET]
X_test = test.drop(columns=["Index", "timestamp"])

# Align columns (test may lack target encoding cols if they matched nothing)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print("Total features:", X.shape[1])
print(X.columns.tolist())

# ==========================================================
# CATBOOST — TUNED HYPERPARAMS
# ==========================================================

cb_cat_features = [
    X.columns.get_loc(c) for c in
    ["geohash", "geohash4", "geohash5", "geohash6",
     "RoadType", "LargeVehicles", "Landmarks", "Weather", "hour_bin"]
    if c in X.columns
]

scores_cb, oof_cb = [], np.zeros(len(X))
test_preds_cb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\n{'='*50}\nCatBoost FOLD {fold+1}\n{'='*50}")

    model_cb = CatBoostRegressor(
        iterations=10000,       # was 5000 — still stopping early at 4999
        learning_rate=0.01,     # halved → smoother convergence
        depth=8,                # reduced from 10 → less overfit
        l2_leaf_reg=5,          # was default 3 → more regularisation
        loss_function="RMSE",
        eval_metric="R2",
        random_seed=42,
        verbose=1000,
        early_stopping_rounds=300,  # stop if no improvement for 300 rounds
    )

    model_cb.fit(
        X.iloc[tr_idx], y.iloc[tr_idx],
        cat_features=cb_cat_features,
        eval_set=(X.iloc[val_idx], y.iloc[val_idx]),
        use_best_model=True,
    )

    preds = model_cb.predict(X.iloc[val_idx])
    fold_r2 = r2_score(y.iloc[val_idx], preds)
    scores_cb.append(fold_r2)
    oof_cb[val_idx] = preds
    test_preds_cb += model_cb.predict(X_test) / N_SPLITS
    print(f"Fold {fold+1} R2 = {fold_r2:.6f}")

print(f"\nCatBoost OOF R2: {r2_score(y, oof_cb):.6f}")

# ==========================================================
# LIGHTGBM — BLEND PARTNER
# ==========================================================

# LightGBM needs label-encoded categoricals
from sklearn.preprocessing import LabelEncoder

X_lgb      = X.copy()
X_test_lgb = X_test.copy()

lgb_cat_cols = ["geohash", "geohash4", "geohash5", "geohash6",
                "RoadType", "LargeVehicles", "Landmarks", "Weather", "hour_bin"]
lgb_cat_cols = [c for c in lgb_cat_cols if c in X_lgb.columns]

for col in lgb_cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X_lgb[col], X_test_lgb[col]]).astype(str)
    le.fit(combined)
    X_lgb[col]      = le.transform(X_lgb[col].astype(str))
    X_test_lgb[col] = le.transform(X_test_lgb[col].astype(str))

scores_lgb, oof_lgb = [], np.zeros(len(X_lgb))
test_preds_lgb = np.zeros(len(X_test_lgb))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_lgb)):
    print(f"\n{'='*50}\nLightGBM FOLD {fold+1}\n{'='*50}")

    model_lgb = LGBMRegressor(
        n_estimators=10000,
        learning_rate=0.01,
        num_leaves=127,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1,
    )

    model_lgb.fit(
        X_lgb.iloc[tr_idx], y.iloc[tr_idx],
        eval_set=[(X_lgb.iloc[val_idx], y.iloc[val_idx])],
        callbacks=[
            __import__("lightgbm").early_stopping(300, verbose=False),
            __import__("lightgbm").log_evaluation(1000),
        ],
        categorical_feature=lgb_cat_cols,
    )

    preds = model_lgb.predict(X_lgb.iloc[val_idx])
    fold_r2 = r2_score(y.iloc[val_idx], preds)
    scores_lgb.append(fold_r2)
    oof_lgb[val_idx] = preds
    test_preds_lgb += model_lgb.predict(X_test_lgb) / N_SPLITS
    print(f"Fold {fold+1} R2 = {fold_r2:.6f}")

print(f"\nLightGBM OOF R2: {r2_score(y, oof_lgb):.6f}")

# ==========================================================
# OPTIMAL BLEND WEIGHT (search on OOF)
# ==========================================================

best_w, best_r2 = 0.5, 0.0
for w in np.arange(0.0, 1.01, 0.02):
    blended = w * oof_cb + (1 - w) * oof_lgb
    r2 = r2_score(y, blended)
    if r2 > best_r2:
        best_r2, best_w = r2, w

print(f"\nBest blend weight (CatBoost) : {best_w:.2f}")
print(f"Blended OOF R2               : {best_r2:.6f}")

test_preds_final = best_w * test_preds_cb + (1 - best_w) * test_preds_lgb

# ==========================================================
# SUBMISSION
# ==========================================================

submission = pd.DataFrame({"Index": test_ids, "demand": test_preds_final})
submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv saved.")
print(submission.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.1 MB/s eta 0:00:00
Target encoding done. New features: ['RoadType_te', 'RoadType_x_hour_te', 'RoadType_x_day_te', 'geohash4_te', 'geohash4_x_hour_te', 'geohash5_te', 'Weather_x_hour_te', 'location_cluster_x_hour_te']
Aggregation features done.
Total features: 44
['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'hour', 'minute', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'time_block', 'latitude', 'longitude', 'geohash4', 'geohash5', 'geohash6', 'temp_sq', 'lane_temp', 'hour_bin', 'geohash_freq', 'geohash4_freq', 'geohash5_freq', 'geohash6_freq', 'location_cluster', 'RoadType_te', 'RoadType_x_hour_te', 'RoadType_x_day_te', 'geohash4_te', 'geohash4_x_hour_te', 'geohash5_te', 'Weather_x_hour_te', 'location_cluster_x_hour_te', 'geohash_hour_mean', 'geohash_hour_std', 'geohash4_hour_mean', 'geohash4_hour_std', 'RoadType_hour_mean', 'RoadType_day_hour_mean', 'location_cluster_h